# 21. SBI Model Selection

Per-animal BE vs SC model assignment using amortised SNPE,
conditioned on expert uniform sessions.

SNPE is trained on heuristic stats only (no UM, no conditional psych).
Model comparison is scored via UM MSE or conditional_psych MSE
from the same posterior.

In [ ]:
%matplotlib inline
from shared_setup import *
import pickle

In [ ]:
# ── Mode ──────────────────────────────────────────────────────────────────
# 'load' = load cluster results from results/
# 'run'  = quick local execution (small settings)
MODE = 'load'

if MODE == 'run':
    N_SYNTHETIC = 3
    BURN_IN = 500
    N_SBI_SIMS = 1_000
    N_CV_REPEATS = 4
    SEED = 42
    print('MODE: run (quick local)')
else:
    print('MODE: load (cluster results)')

## 1. Load / Run SBI Results

In [ ]:
sbi_results = {}  # {fit_target: DataFrame}

if MODE == 'load':
    for ft in FIT_TARGETS:
        comp_dir = SBI_STATIC_DIR / 'comparisons' / f'uniform_{ft}'
        if not comp_dir.exists():
            print(f'Missing: {comp_dir}')
            print('  Run: python scripts/condition_sbi_local.py --distribution uniform')
            continue
        files = sorted(comp_dir.glob('animal_*.pkl'))
        if not files:
            print(f'No results in {comp_dir}')
            continue
        rows = []
        for p in files:
            with open(p, 'rb') as f:
                rows.append(pickle.load(f))
        sbi_results[ft] = pd.DataFrame(rows)
        print(f'Loaded {ft}: {len(rows)} animals')

elif MODE == 'run':
    try:
        import torch
    except ImportError:
        raise RuntimeError('SBI requires torch — install it or use MODE=load')

    from behav_utils.data.selection import fitting_data_from_sessions
    from inference.comparison import train_amortised_snpe, run_animal_pipeline
    from scripts.config import SBI_STATS

    experiment, _ = load_data()
    be_snpe = train_amortised_snpe('be', SBI_STATS, N_SBI_SIMS, 300, BURN_IN, SEED)
    sc_snpe = train_amortised_snpe('sc', SBI_STATS, N_SBI_SIMS, 300, BURN_IN, SEED+1)

    animal_ids = sorted(experiment.animals.keys())[:N_SYNTHETIC]
    for ft in FIT_TARGETS:
        rows = []
        for aid in animal_ids:
            sessions = select_sessions(experiment.animals[aid], 'expert_uniform')
            if not sessions: continue
            fd = fitting_data_from_sessions(sessions, aid)
            r = run_animal_pipeline(fd, be_snpe, sc_snpe,
                                   n_cv_repeats=N_CV_REPEATS, seed=SEED,
                                   method=ft)
            rows.append(r)
        sbi_results[ft] = pd.DataFrame(rows)
        print(f'{ft}: {len(rows)} animals')

## 2. Per-Animal Assignment

In [ ]:
for ft, df in sbi_results.items():
    if len(df) == 0: continue
    print(f'\n=== {ft} ===')
    cols = ['animal_id', 'winner', 'p', 'be_mean', 'sc_mean']
    cols = [c for c in cols if c in df.columns]
    print(df[cols].to_string(index=False, float_format='%.5f'))
    if 'winner' in df.columns:
        print(f'\nAssignment: {df["winner"].value_counts().to_dict()}')

## 3. Error Comparison

In [ ]:
from behav_utils.plotting.styles import COLOURS
BE_COL, SC_COL = COLOURS['BE'], COLOURS['SC']

for ft, df in sbi_results.items():
    if len(df) == 0 or 'be_mean' not in df.columns: continue
    fig, ax = plt.subplots(figsize=(10, 5))
    aid_col = 'animal_id' if 'animal_id' in df.columns else 'animal'
    x = np.arange(len(df))
    w = 0.35
    ax.bar(x - w/2, df['be_mean'], w, label='BE', color=BE_COL, alpha=0.7)
    ax.bar(x + w/2, df['sc_mean'], w, label='SC', color=SC_COL, alpha=0.7)
    ax.set_xticks(x)
    ax.set_xticklabels(df[aid_col], rotation=45, ha='right')
    ax.set_ylabel('Mean CV Error (MSE)')
    ax.set_title(f'SBI Model Selection — {ft}')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 4. Fit Target Comparison

In [ ]:
if len(sbi_results) == 2:
    um = sbi_results.get('update_matrix', pd.DataFrame())
    cp = sbi_results.get('conditional_psych', pd.DataFrame())
    aid_col = 'animal_id' if 'animal_id' in um.columns else 'animal'
    if len(um) > 0 and len(cp) > 0:
        merged = um[[aid_col, 'winner']].rename(columns={'winner': 'um_winner'}).merge(
            cp[[aid_col, 'winner']].rename(columns={'winner': 'cp_winner'}),
            on=aid_col, how='inner',
        )
        merged['agree'] = merged['um_winner'] == merged['cp_winner']
        print(merged.to_string(index=False))
        print(f'\nAgreement: {merged["agree"].mean():.0%}')
else:
    print('Only one fit target available.')